# Lab 2 - RAG ve Getirme Kalitesi

**Kanitladigi tez:** Sorun genellikle modelde degil, ona ne verdiginizdedir.

Bu labda model **hic degismeyecek**. Yalnizca modele verdigimiz baglami
degistirecegiz ve cevabin bozulup duzeldigini gorecegiz.

## Kurulum

Asagidaki iki hucreyi sirayla calistirin. Bilgisayariniza hicbir sey
kurulmuyor: her sey sizin Colab calisma zamaninizda calisir ve
oturum kapaninca silinir.

In [ ]:
# 1/2 - Repoyu indir
!git clone -q https://github.com/silexi/guvenli-ai-mimarileri-lab.git 2>/dev/null || echo 'repo zaten var'
%cd -q guvenli-ai-mimarileri-lab
!pip install -q -U transformers accelerate 2>/dev/null
print('kurulum tamam')

In [ ]:
# 2/2 - Modeli sec ve yukle
import os, sys
sys.path.insert(0, '.')

# Uc secenek:
#   'colab' -> kendi calisma zamaninizda kucuk bir model (varsayilan)
#   'mock'  -> model yuklemeden, kayitli cevaplarla (yedek yol)
os.environ['LAB_SAGLAYICI'] = 'colab'

from ortak import llm
print(llm.durum())

# Modeli simdi yukleyelim ki sonraki hucreler beklemesin.
# GPU yoksa bu adim birkac dakika surebilir.
try:
    llm.model_yukle()
except Exception as hata:
    print('Model yuklenemedi:', hata)
    print('MOCK moda geciliyor, lab yapisi aynen calisacak.')
    os.environ['LAB_SAGLAYICI'] = 'mock'

print('\nLab 2 icin hazir.')

---
## 1. Bilgi bankasini kuralim

`data/docs` altinda yedi kisa politika belgesi var. Dosya adinin
onundeki etiket gizlilik seviyesini belirliyor: `genel_`, `ic_`, `gizli_`.

In [ ]:
from ortak import rag

parcalar = rag.belgeleri_yukle('data/docs', strateji='paragraf')
print(f'{len(parcalar)} parca olustu.')
print()
for p in parcalar[:5]:
    print(f'  {p.kaynak} [{p.gizlilik}] - {p.metin[:60]}...')

bb = rag.BilgiBankasi(parcalar)

---
## 2. Normal bir RAG sorgusu

Once getirme adimini **gorunur** kiliyoruz. Cogu RAG hatasi burada
gizlidir ama kimse bakmadigi icin modele fatura kesilir.

In [ ]:
SORU = 'Fatura itirazi kac gun icinde incelenir?'

getirilen = bb.ara(SORU, k=3)
rag.parcalari_goster(getirilen)

In [ ]:
from ortak import llm

def cevapla(soru, parcalar, senaryo=None):
    baglam = rag.baglam_kur(parcalar)
    istek = f'''Asagidaki baglami kullanarak soruyu cevapla.
Baglamda olmayan bir sey uydurma.

BAGLAM:
{baglam}

SORU: {soru}'''
    return llm.sor(istek, sicaklik=0.3, senaryo=senaryo), istek

cevap, istek = cevapla(SORU, getirilen, senaryo='lab2_iyi')
print('CEVAP:')
print(cevap.metin)
print()
print(f'Girdi token: {cevap.girdi_token}')

---
## 3. Baglama alakasiz bir parca sokalim

Model ayni. Prompt ayni. Yalnizca baglama, soruyla ilgisi olmayan
bir parca ekliyoruz.

In [ ]:
# Iade politikasindan bir parca -- soru fatura itirazi hakkindaydi
alakasiz = [p for p in parcalar if 'iade' in p.kaynak][0]
print('Sokulan alakasiz parca:')
print(f'  {alakasiz.kaynak}: {alakasiz.metin[:120]}...')

kirli_baglam = getirilen[:2] + [alakasiz]
rag.parcalari_goster(kirli_baglam, 'Kirletilmis baglam')

In [ ]:
cevap_kirli, _ = cevapla(SORU, kirli_baglam, senaryo='lab2_kotu')
print('CEVAP (kirletilmis baglamla):')
print(cevap_kirli.metin)

> **Ne oldu?** Cevap artik soruda hic sorulmayan iade ve indirim
> bilgisini icerebilir. Model kotu degil; ona verdiginiz baglam kotu.
> DoorDash'in yayinladigi mimaride de kok sebep tam olarak buydu:
> getirilen baglam belirsiz kalinca model egitim verisine donup
> var olmayan politikalar uydurmus.

---
## 4. Parcalama (chunking) stratejisi her seyi belirler

Simdi ayni belgeleri **kotu** bir stratejiyle parcalayalim:
anlam butunlugune bakmadan, sabit karakter sayisiyla.

In [ ]:
kotu_parcalar = rag.belgeleri_yukle('data/docs', strateji='sabit')
kotu_bb = rag.BilgiBankasi(kotu_parcalar)

print(f'Iyi strateji : {len(parcalar)} parca')
print(f'Kotu strateji: {len(kotu_parcalar)} parca')
print()

kotu_getirilen = kotu_bb.ara(SORU, k=3)
rag.parcalari_goster(kotu_getirilen, 'Kotu parcalama ile getirilen')

> Cumlelerin ortasindan bolunmus parcalara dikkat edin. Bir kural
> parcanin sonunda kesilmisse, model o kuralin yarisini gorur ve
> kalan yarisini kendi tamamlar.

---
## 5. Yetki filtresi: getirmeden ONCE mi, sonra mi?

Bu, 1. gunde 'en cok atlanan madde' dedigimiz noktaydi. Bilgi bankasinda
`gizli_fiyatlandirma.md` var. Genel yetkili bir kullanici bunu
gormemeli.

In [ ]:
GIZLI_SORU = 'Kurumsal musteriler icin taban maliyet carpani nedir?'

print('### Filtresiz (yanlis) ###')
filtresiz = bb.ara(GIZLI_SORU, k=2)
rag.parcalari_goster(filtresiz)

In [ ]:
print('### Yetki filtresi getirmeden ONCE uygulanmis (dogru) ###')
filtreli = bb.ara(GIZLI_SORU, k=2, kullanici_gizlilik_seviyesi='genel')
rag.parcalari_goster(filtreli)

In [ ]:
# Filtresiz baglamla cevap uretirsek ne olur?
cevap_sizinti, _ = cevapla(GIZLI_SORU, filtresiz,
                           senaryo='lab2_yetkisiz')
print('Filtresiz baglamla uretilen cevap:')
print(cevap_sizinti.metin)
print()
print('>>> Bu bir veri sizintisidir. Model kusurlu davranmadi;',
      'ona gormemesi gereken veriyi biz verdik.')

> **Neden sirasi onemli?** Once getirip sonra filtrelerseniz, gizli
> icerik zaten baglama girmis olur. Modelden 'bunu soyleme' diye rica
> etmek bir kilit degildir. Filtre, getirme sorgusunun parcasi olmali.

---
## Egzersiz

1. `k` degerini 3'ten 8'e cikarin. Cevap iyilesiyor mu, bozuluyor mu?
   Girdi token sayisi ne kadar artiyor?
2. `kullanici_gizlilik_seviyesi='ic'` yapin. Hangi belgeler geliyor?
3. `data/docs` altina kendi kurumunuzdan (anonimlestirilmis) bir politika
   belgesi ekleyin ve kendi sorunuzu sorun.

---
## Alinacak ders

> Getirme kalitesi = sistem kalitesi. Model bu labda hic degismedi;
> cevabi bozan da duzelten de ona ne verdigimizdi.